# Notebook 2: Baseline Model Bake-Off

Compare five non-ML prediction baselines:
1. **Random** (Gaussian noise)
2. **Persistence** (predict no movement)
3. **Linear extrapolation** (least-squares velocity fit)
4. **Kernel regression** (Gaussian-weighted neighbor lookup)
5. **Branching particle filter** (direction-aware, multi-modal)

We also diagnose the linear extrapolation model, which performs suspiciously poorly.

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import OrderedDict

REPO_ROOT = "/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 1. Load data (concise)

In [ ]:
BUILD_DIR = "/net/trapnell/vol1/home/mdcolon/proj/morphseq/morphseq_playground/metadata/build06_output"
EXPERIMENT_IDS = ["20251207_pbx", "20251121"]  # use a manageable subset

from dev.dynamo.data import load_trajectories, FragmentDataset, fragment_collate_fn

dataset = load_trajectories(
    experiment_ids=EXPERIMENT_IDS,
    build_dir=BUILD_DIR,
    n_components=10,
    scale=True,
    verbose=True,
)
print(f"\nLoaded {len(dataset)} trajectories, {dataset.n_components} PCs, "
      f"{len(dataset.class_to_idx)} classes")

In [ ]:
# Create evaluation FragmentDataset
eval_frag = FragmentDataset(
    dataset,
    min_context=3,
    horizons=(1, 2, 3, 4),
    epoch_length=2000,
    gamma=0.0,   # no rebalancing for eval
    n_targets=1,
)

## 2. Set up all five predictors

In [ ]:
from dev.dynamo.eval.predictions import (
    GaussianNoisePredictor,
    PersistencePredictor,
    LinearExtrapolationPredictor,
)
from dev.dynamo.models import SimpleKernelPredictor, BranchingParticleFilter

predictors = OrderedDict([
    ("Random",          GaussianNoisePredictor(std=1.0)),
    ("Persistence",     PersistencePredictor(noise_scale=0.1)),
    ("Linear",          LinearExtrapolationPredictor(noise_scale=0.1, n_points=5)),
    ("Kernel",          SimpleKernelPredictor(dataset, n_samples=100, exclude_self=True)),
    ("Particle Filter", BranchingParticleFilter(dataset, n_samples=100, exclude_self=True)),
])

print("Predictors ready:")
for name in predictors:
    print(f"  - {name}")

## 3. Run evaluation

In [ ]:
from dev.dynamo.eval import run_evaluation, EvalResult

results = OrderedDict()
N_BATCHES = 50
BATCH_SIZE = 32

for name, predictor in predictors.items():
    print(f"Evaluating {name}...", end=" ", flush=True)
    result = run_evaluation(
        predictor, eval_frag,
        n_batches=N_BATCHES,
        batch_size=BATCH_SIZE,
    )
    results[name] = result
    print(f"NLL={result.metrics['nll']:.3f}  MSE={result.metrics['mse']:.6f}  "
          f"Cal={result.calibration:.3f}  (n={result.n_samples})")

print("\nDone.")

## 4. Results comparison

In [ ]:
# Summary table
metric_names = ["nll", "mse", "rmse"]
if "energy_distance" in list(results.values())[0].metrics:
    metric_names.append("energy_distance")

print(f"{'Model':<20s}", end="")
for m in metric_names:
    print(f"{m:>14s}", end="")
print(f"{'calibration':>14s}")
print("-" * (20 + 14 * (len(metric_names) + 1)))

for name, res in results.items():
    print(f"{name:<20s}", end="")
    for m in metric_names:
        val = res.metrics.get(m, float('nan'))
        print(f"{val:>14.4f}", end="")
    print(f"{res.calibration:>14.4f}")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = list(results.keys())
x = np.arange(len(model_names))
colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]

for ax, metric in zip(axes, ["nll", "mse", "calibration"]):
    if metric == "calibration":
        vals = [results[n].calibration for n in model_names]
    else:
        vals = [results[n].metrics[metric] for n in model_names]
    bars = ax.bar(x, vals, color=colors, edgecolor="k", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha="right", fontsize=9)
    ax.set_title(metric.upper())
    ax.set_ylabel(metric)
    # Add value labels
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Baseline Model Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### Per-horizon breakdown

In [ ]:
# Per-horizon MSE and NLL
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, res in results.items():
    horizons = sorted(res.per_horizon.keys())
    mse_vals = [res.per_horizon[k]["mse"] for k in horizons]
    nll_vals = [res.per_horizon[k]["nll"] for k in horizons]
    
    axes[0].plot(horizons, mse_vals, "o-", label=name, lw=2, markersize=6)
    axes[1].plot(horizons, nll_vals, "o-", label=name, lw=2, markersize=6)

axes[0].set_xlabel("Horizon k (frames)")
axes[0].set_ylabel("MSE")
axes[0].set_title("MSE by prediction horizon")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Horizon k (frames)")
axes[1].set_ylabel("NLL")
axes[1].set_title("NLL by prediction horizon")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 5. Diagnosing the Linear Extrapolation problem

The linear predictor performs worse than random. Let's figure out why.

**Hypothesis**: The velocity is fit from `time_deltas` (seconds between context frames),
and then extrapolated by `horizon_dt` (also in seconds). If the velocity estimate is
noisy and `horizon_dt` is large (thousands of seconds), small velocity errors get
amplified into massive displacement errors.

In [ ]:
# Collect per-sample diagnostics from the linear predictor
from torch.utils.data import DataLoader
from dev.dynamo.eval.metrics import mse as mse_fn

eval_frag._epoch_length = 1000
loader = DataLoader(eval_frag, batch_size=32, collate_fn=fragment_collate_fn, shuffle=False)

linear_pred = LinearExtrapolationPredictor(noise_scale=0.1, n_points=5)
persist_pred = PersistencePredictor(noise_scale=0.1)

linear_mses = []
persist_mses = []
horizon_dts = []
predicted_displacements = []
actual_displacements = []

with torch.no_grad():
    for batch in loader:
        # Linear
        lin_result = linear_pred.predict(batch)
        per_result = persist_pred.predict(batch)
        target = batch.target  # (B, D)
        
        linear_mses.append(mse_fn(lin_result.predicted_mean, target))
        persist_mses.append(mse_fn(per_result.predicted_mean, target))
        horizon_dts.append(batch.horizon_dt)
        
        # How far did the linear predictor think things would move?
        last_frame = per_result.predicted_mean  # persistence = last context frame
        predicted_displacements.append(
            (lin_result.predicted_mean - last_frame).norm(dim=-1))
        actual_displacements.append(
            (target - last_frame).norm(dim=-1))

linear_mses = torch.cat(linear_mses).numpy()
persist_mses = torch.cat(persist_mses).numpy()
horizon_dts = torch.cat(horizon_dts).numpy()
pred_disp = torch.cat(predicted_displacements).numpy()
actual_disp = torch.cat(actual_displacements).numpy()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. MSE vs horizon_dt
ax = axes[0, 0]
ax.scatter(horizon_dts, linear_mses, s=5, alpha=0.3, label="Linear", color="green")
ax.scatter(horizon_dts, persist_mses, s=5, alpha=0.3, label="Persistence", color="orange")
ax.set_xlabel("horizon_dt (seconds)")
ax.set_ylabel("MSE")
ax.set_title("MSE vs prediction time gap")
ax.legend(fontsize=8)
ax.set_yscale("log")

# 2. Predicted vs actual displacement
ax = axes[0, 1]
ax.scatter(actual_disp, pred_disp, s=5, alpha=0.3, color="green")
lim = max(actual_disp.max(), pred_disp.max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", alpha=0.5, label="y=x")
ax.set_xlabel("Actual displacement ||target - last_frame||")
ax.set_ylabel("Predicted displacement (linear)")
ax.set_title("Linear: predicted vs actual displacement")
ax.legend()

# 3. Distribution of predicted displacements
ax = axes[1, 0]
ax.hist(actual_disp, bins=50, alpha=0.5, label="Actual", color="gray", edgecolor="k")
ax.hist(pred_disp, bins=50, alpha=0.5, label="Linear predicted", color="green", edgecolor="k")
ax.set_xlabel("Displacement magnitude")
ax.set_ylabel("Count")
ax.set_title("Displacement distributions")
ax.legend()
ax.set_yscale("log")

# 4. Ratio of linear MSE to persistence MSE
ax = axes[1, 1]
ratio = linear_mses / np.maximum(persist_mses, 1e-10)
ax.hist(np.clip(ratio, 0, 20), bins=50, edgecolor="k", alpha=0.7, color="steelblue")
ax.axvline(1.0, color="red", ls="--", label="Linear = Persistence")
ax.set_xlabel("MSE ratio (Linear / Persistence)")
ax.set_ylabel("Count")
ax.set_title(f"Linear/Persistence MSE ratio\nmedian={np.median(ratio):.2f}, "
             f">1 in {(ratio > 1).mean()*100:.1f}% of samples")
ax.legend()

plt.suptitle("Diagnosing the Linear Extrapolation problem", fontsize=14)
plt.tight_layout()
plt.show()

### Inspect worst-case linear predictions

Look at the samples where linear extrapolation is most catastrophically wrong.

In [ ]:
# Rerun on a small set to capture full sample details
eval_frag._epoch_length = 200
loader = DataLoader(eval_frag, batch_size=200, collate_fn=fragment_collate_fn, shuffle=False)
batch = next(iter(loader))

lin_result = linear_pred.predict(batch)
per_result = persist_pred.predict(batch)
target = batch.target

lin_err = mse_fn(lin_result.predicted_mean, target).numpy()
per_err = mse_fn(per_result.predicted_mean, target).numpy()

# Find worst cases for linear (high MSE) where persistence does fine
ratio = lin_err / np.maximum(per_err, 1e-10)
worst_idx = np.argsort(ratio)[-6:]  # top 6 worst

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for panel, idx in enumerate(worst_idx):
    ax = axes[panel]
    
    # Extract context for this sample
    L = batch.context_mask[idx].sum().item()
    ctx = batch.context[idx, :L].numpy()
    tgt = target[idx].numpy()
    lin_pred_pt = lin_result.predicted_mean[idx].numpy()
    per_pred_pt = per_result.predicted_mean[idx].numpy()
    
    # Also get parent trajectory
    traj_idx = batch.embryo_idx[idx].item()
    parent = dataset.trajectories[traj_idx]
    
    pc_x, pc_y = 0, 1
    
    # Parent trajectory
    ax.plot(parent.trajectory[:, pc_x], parent.trajectory[:, pc_y],
            color="lightgray", lw=1, zorder=1)
    
    # Context
    ax.plot(ctx[:, pc_x], ctx[:, pc_y], "b-", lw=2, zorder=3, label="context")
    ax.scatter(ctx[-1, pc_x], ctx[-1, pc_y], color="blue", s=60, zorder=4, marker=">")
    
    # Target
    ax.scatter(tgt[pc_x], tgt[pc_y], color="gold", s=120, zorder=5,
               marker="D", edgecolors="k", linewidths=1, label="target")
    
    # Persistence prediction
    ax.scatter(per_pred_pt[pc_x], per_pred_pt[pc_y], color="orange", s=80, zorder=5,
               marker="o", edgecolors="k", linewidths=0.5, label=f"persist (MSE={per_err[idx]:.4f})")
    
    # Linear prediction (may be far off!)
    ax.scatter(lin_pred_pt[pc_x], lin_pred_pt[pc_y], color="green", s=80, zorder=5,
               marker="^", edgecolors="k", linewidths=0.5, label=f"linear (MSE={lin_err[idx]:.4f})")
    
    # Arrow from last context to linear prediction
    ax.annotate("", xy=(lin_pred_pt[pc_x], lin_pred_pt[pc_y]),
                xytext=(ctx[-1, pc_x], ctx[-1, pc_y]),
                arrowprops=dict(arrowstyle="->", color="green", lw=1.5, alpha=0.6))
    
    dt = batch.horizon_dt[idx].item()
    ax.set_title(f"Sample {idx}: horizon_dt={dt:.0f}s\n"
                 f"ratio={ratio[idx]:.1f}x worse than persistence", fontsize=9)
    ax.legend(fontsize=6, loc="best")
    ax.set_xlabel(f"PC1", fontsize=8)
    ax.set_ylabel(f"PC2", fontsize=8)

plt.suptitle("Worst-case Linear Extrapolation predictions\n"
             "(green arrow shows where linear extrap goes)", fontsize=13)
plt.tight_layout()
plt.show()

### Root cause analysis

Let's quantify: is it the velocity magnitude, the direction, or the horizon_dt scaling?

In [ ]:
# Compute velocity norms from the linear predictor manually
B = batch.context.shape[0]
D = batch.context.shape[2]
lengths = batch.context_mask.sum(dim=1).long()

velocities = []
for b in range(B):
    L = lengths[b].item()
    n_use = min(5, L)
    if n_use < 2:
        velocities.append(np.zeros(D))
        continue
    start = L - n_use
    frames = batch.context[b, start:L].numpy()
    deltas = batch.time_deltas[b, start:L-1].numpy()
    t = np.zeros(n_use)
    t[1:] = np.cumsum(deltas)
    t_mean = t.mean()
    t_c = t - t_mean
    var_t = (t_c ** 2).sum()
    z_mean = frames.mean(axis=0)
    z_c = frames - z_mean
    cov_tz = (t_c[:, None] * z_c).sum(axis=0)
    velocity = cov_tz / max(var_t, 1e-8)
    velocities.append(velocity)

velocities = np.array(velocities)
v_norms = np.linalg.norm(velocities, axis=1)
h_dts = batch.horizon_dt.numpy()

# Predicted displacement = ||velocity|| * horizon_dt
pred_displacement = v_norms * h_dts

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(v_norms, bins=50, edgecolor="k", alpha=0.7, color="steelblue")
axes[0].set_xlabel("||velocity|| (PC units / second)")
axes[0].set_title("Fitted velocity magnitude")

axes[1].hist(h_dts, bins=50, edgecolor="k", alpha=0.7, color="salmon")
axes[1].set_xlabel("horizon_dt (seconds)")
axes[1].set_title("Prediction time gap")

axes[2].hist(np.clip(pred_displacement, 0, np.percentile(pred_displacement, 99)),
             bins=50, edgecolor="k", alpha=0.7, color="green")
axes[2].axvline(np.median(actual_disp[:B]), color="red", ls="--",
                label=f"median actual disp = {np.median(actual_disp[:B]):.3f}")
axes[2].set_xlabel("Predicted displacement (||v|| * horizon_dt)")
axes[2].set_title("Linear predicted displacement magnitude")
axes[2].legend(fontsize=8)

plt.suptitle("Linear extrapolation: velocity * horizon_dt analysis", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Velocity ||v||: median={np.median(v_norms):.6f}, "
      f"mean={v_norms.mean():.6f}, max={v_norms.max():.6f}")
print(f"horizon_dt:    median={np.median(h_dts):.0f}s, "
      f"mean={h_dts.mean():.0f}s, max={h_dts.max():.0f}s")
print(f"Predicted displacement: median={np.median(pred_displacement):.3f}, "
      f"max={pred_displacement.max():.3f}")
print(f"Actual displacement:    median={np.median(actual_disp[:B]):.3f}, "
      f"max={actual_disp[:B].max():.3f}")

### Verdict

The linear extrapolation predictor multiplies an estimated velocity vector (in PC-units/second) by `horizon_dt` (in seconds). Even small velocity errors get amplified by large time gaps.

The fundamental issue is that biological trajectories **curve and slow down** — they don't follow straight lines. Extrapolating a linear trend over thousands of seconds overshoots badly.

This is actually expected behavior: the linear predictor is a worse model of embryo dynamics than even "predict no movement" for multi-frame horizons. This validates the need for nonlinear baselines (kernel, particle filter) and the learned potential model.

---
## 6. Visual comparison: sample predictions

Show predictions from all models side-by-side on the same samples.

In [ ]:
# Collect predictions from all models on the same batch
eval_frag._epoch_length = 100
loader = DataLoader(eval_frag, batch_size=100, collate_fn=fragment_collate_fn, shuffle=False)
batch = next(iter(loader))

all_preds = {}
for name, predictor in predictors.items():
    all_preds[name] = predictor.predict(batch)

# Pick 6 random samples to visualize
rng = np.random.default_rng(42)
show_idx = rng.choice(batch.context.shape[0], size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
model_markers = {"Random": "x", "Persistence": "o", "Linear": "^",
                 "Kernel": "s", "Particle Filter": "P"}
model_colors = {"Random": "#d62728", "Persistence": "#ff7f0e", "Linear": "#2ca02c",
                "Kernel": "#1f77b4", "Particle Filter": "#9467bd"}

for panel, idx in enumerate(show_idx):
    ax = axes[panel]
    L = batch.context_mask[idx].sum().item()
    ctx = batch.context[idx, :L].numpy()
    tgt = batch.target[idx].numpy()
    traj_idx = batch.embryo_idx[idx].item()
    parent = dataset.trajectories[traj_idx]
    pc_x, pc_y = 0, 1
    
    # Parent trajectory
    ax.plot(parent.trajectory[:, pc_x], parent.trajectory[:, pc_y],
            color="lightgray", lw=1, zorder=1)
    # Context
    ax.plot(ctx[:, pc_x], ctx[:, pc_y], "k-", lw=2.5, zorder=3)
    ax.scatter(ctx[-1, pc_x], ctx[-1, pc_y], color="k", s=50, zorder=4, marker=">")
    # Target
    ax.scatter(tgt[pc_x], tgt[pc_y], color="gold", s=150, zorder=6,
               marker="D", edgecolors="k", linewidths=1.5)
    
    # Model predictions
    for name in predictors:
        pred_pt = all_preds[name].predicted_mean[idx].numpy()
        ax.scatter(pred_pt[pc_x], pred_pt[pc_y],
                   color=model_colors[name], s=60, zorder=5,
                   marker=model_markers[name], edgecolors="k", linewidths=0.5,
                   label=name if panel == 0 else "")
    
    dt = batch.horizon_dt[idx].item()
    ax.set_title(f"horizon_dt={dt:.0f}s ({dt/parent.delta_t:.1f} frames)", fontsize=9)
    ax.set_xlabel("PC1", fontsize=8)
    ax.set_ylabel("PC2", fontsize=8)

axes[0].legend(fontsize=7, loc="best")
plt.suptitle("Model predictions (gold diamond = target, black = context)", fontsize=13)
plt.tight_layout()
plt.show()